In [ ]:
# -*- coding: utf-8 -*-
# GA for NIPT 5-bin scheduling with HARD constraint
# Model (no interaction): Y = b0 + bg*GA + bb*BMI + ba*Age + u + eps
import numpy as np, pandas as pd
from math import sqrt
from scipy.stats import norm

# ===== User config =====
b0, bg, bb = 0.0895180, 0.0028460, -0.0011022
ba = -0.0007929           # <-- TODO: 用你们LMM估计值替换
sig2, sigu2 = 0.01566**2, 0.02643**2
THRESH = 0.04
PROB_REQ = 0.8        # 硬性约束：组内 P>=0.8 的比例 ≥ 0.8

# Risk pieces
def W(t):  # window risk（示例Sigmoid）
    return 1 / (1 + np.exp(-0.45*(t-17)))
LAM = 0.7
use_delay = False
Delta = 1.5

# GA settings
K = 5
t_min, t_max, t_step = 10.0, 25.0, 0.01
T_GRID = np.round(np.arange(t_min, t_max+1e-9, t_step), 4)
min_group_size = 30
POP, GEN = 120, 150
pc, pm = 0.5, 0.2
elite = 4

# ===== Data: provide BMI (for grouping) and Age (only for model) =====
try:
    df = pd.read_csv("dabiao_male.csv")
    # BMI
    for col in ["BMI","孕妇BMI"]:
        if col in df.columns:
            BMI = df[col].astype(float).values
            break
    else:
        raise ValueError("No BMI column found.")
    # Age
    for col in ["Age","孕妇年龄","年龄"]:
        if col in df.columns:
            AGE = df[col].astype(float).values
            break
    else:
        raise ValueError("No Age column found.")
except Exception:
    rng = np.random.default_rng(42)
    BMI = rng.normal(26.0, 4.0, size=200).clip(16, 42)  # demo BMI
    AGE = rng.normal(30.0, 4.0, size=200).clip(18, 45)  # demo Age

assert len(BMI) == len(AGE), "BMI 与 Age 数组长度必须一致"

# ===== Core probabilistic model =====
def mu(b, t, a):
    return b0 + bg*t + bb*b + ba*a

def p_gt(b, t, a):
    # P(Y > THRESH | BMI=b, GA=t, Age=a) under Normal with RE variance
    sd = sqrt(sig2 + sigu2)
    z = (THRESH - mu(b, t, a)) / sd
    return 1 - norm.cdf(z)

def risk(b, t, a):
    p = p_gt(b, t, a)
    inv = 1 - p
    base = LAM*inv + 0.3*W(t)
    if use_delay:
        base += (W(t+Delta) - W(t)) * inv
    return base

# ===== Precompute tables (sorted by BMI, Age aligned!) =====
order = np.argsort(BMI)
BMI_sorted = BMI[order]
AGE_sorted = AGE[order]
n = len(BMI_sorted)

Ptab = np.empty((n, len(T_GRID)))
Rtab = np.empty((n, len(T_GRID)))
for i, (b, a) in enumerate(zip(BMI_sorted, AGE_sorted)):
    # 这里逐时点计算，确保 Age 进入概率与风险
    for j, t in enumerate(T_GRID):
        pp = p_gt(b, t, a)
        Ptab[i, j] = pp
        if use_delay:
            Rtab[i, j] = LAM*(1-pp) + 0.3*W(t) + (W(t+Delta)-W(t))*(1-pp)
        else:
            Rtab[i, j] = LAM*(1-pp) + 0.3*W(t)

# ===== Chromosome = (cut_idx[4], time_idx[5]) =====
def random_chrom():
    cuts_base = np.linspace(0, n, K+1).astype(int)
    cuts = cuts_base[1:-1].copy()
    jitter = max(1, n//(5*4))
    cuts = np.clip(cuts + np.random.randint(-jitter, jitter+1, size=4), 0, n)
    cuts.sort()
    times = np.random.randint(0, len(T_GRID), size=K)
    return (cuts, times)

def decode_groups(cuts):
    idx = np.concatenate(([0], cuts, [n]))
    return [(idx[k], idx[k+1]) for k in range(K)]

def repair(cuts):
    idx = np.concatenate(([0], cuts.copy(), [n]))
    for k in range(K):
        s, e = idx[k], idx[k+1]
        if e - s < min_group_size:
            need = min_group_size - (e - s)
            if k < K-1 and (idx[k+2]-idx[k+1]-need) >= min_group_size:
                idx[k+1] += need
            elif k > 0 and (idx[k]-idx[k-1]-need) >= min_group_size:
                idx[k] -= need
            else:
                return None
    new_cuts = idx[1:-1]
    if np.any(np.diff(new_cuts) < 0):
        new_cuts.sort()
    return new_cuts

# ===== HARD feasibility (BMI-only grouping; Age only affects P/R) =====
def best_feasible_time(s, e, j):
    """
    硬性约束：在组 [s:e) 内，满足 P>=0.8 的“个体比例” >= 0.8 才可行。
    若当前 j 不可行，则在全网格中寻找满足硬约束且均值风险最小的 j'。
    不存在则返回 None。
    """
    prop_ok = (Ptab[s:e, j] >= PROB_REQ).mean() >= PROB_REQ
    if prop_ok:
        return j
    feas = np.where((Ptab[s:e, :] >= PROB_REQ).mean(axis=0) >= PROB_REQ)[0]
    if feas.size == 0:
        return None
    j_star = feas[np.argmin(Rtab[s:e, feas].mean(axis=0))]
    return j_star

def fitness(chrom):
    """
    返回 (目标值, chosen_times)
    任一组无可行时点 -> (1e9, None)
    """
    cuts, times = chrom
    c = repair(cuts)
    if c is None:
        return 1e9, None

    segs = decode_groups(c)
    total = 0.0
    chosen = []

    for k, (s, e) in enumerate(segs):
        j = times[k]
        if not (0 <= j < len(T_GRID)):
            return 1e9, None
        j_use = best_feasible_time(s, e, j)
        if j_use is None:
            return 1e9, None
        chosen.append(j_use)
        total += Rtab[s:e, j_use].sum()

    return total / n, np.array(chosen, dtype=int)

# ===== GA operators =====
def tournament(pop, k=3):
    pick = np.random.randint(0, len(pop), size=k)
    best = min(pick, key=lambda i: pop[i][2])
    return pop[best][0].copy(), pop[best][1].copy()

def cx(c1, c2):
    cuts1, t1 = c1; cuts2, t2 = c2
    if np.random.rand() < 0.5:
        pt = np.random.randint(1, 4)
        nc1 = np.concatenate([cuts1[:pt], cuts2[pt:]])
        nc2 = np.concatenate([cuts2[:pt], cuts1[pt:]])
    else:
        nc1, nc2 = cuts1.copy(), cuts2.copy()
    if np.random.rand() < 0.5:
        pt = np.random.randint(1, 5)
        nt1 = np.concatenate([t1[:pt], t2[pt:]])
        nt2 = np.concatenate([t2[:pt], t1[pt:]])
    else:
        nt1, nt2 = t1.copy(), t2.copy()
    nc1.sort(); nc2.sort()
    return (nc1, nt1), (nc2, nt2)

def mut(chrom, p_cut=0.3, p_time=0.5):
    cuts, t = chrom
    cuts = cuts.copy(); t = t.copy()
    for i in range(4):
        if np.random.rand() < p_cut:
            cuts[i] += np.random.randint(-max(1, n//50), max(2, n//50)+1)
    cuts = np.clip(cuts, 0, n); cuts.sort()
    for k in range(K):
        if np.random.rand() < p_time:
            if np.random.rand() < 0.7:
                t[k] += np.random.choice([-1, 1])
            else:
                t[k] = np.random.randint(0, len(T_GRID))
    t = np.clip(t, 0, len(T_GRID)-1)
    return (cuts, t)

# ===== Main GA loop =====
def run_ga():
    pop = []
    for _ in range(POP):
        cuts, t = random_chrom()
        f, chosen = fitness((cuts, t))
        pop.append([cuts, t, f, chosen])

    for g in range(GEN):
        pop.sort(key=lambda x: x[2])
        new = pop[:elite]  # elitism

        while len(new) < POP:
            p1_cuts, p1_times = tournament(pop)
            p2_cuts, p2_times = tournament(pop)
            c1, c2 = ((p1_cuts, p1_times), (p2_cuts, p2_times))

            if np.random.rand() < pc:
                c1, c2 = cx(c1, c2)
            if np.random.rand() < pm:
                c1 = mut(c1)
            if np.random.rand() < pm:
                c2 = mut(c2)

            f1, ch1 = fitness(c1)
            new.append([c1[0], c1[1], f1, ch1])
            if len(new) < POP:
                f2, ch2 = fitness(c2)
                new.append([c2[0], c2[1], f2, ch2])

        pop = new
        if (g + 1) % 20 == 0:
            print(f"Gen {g+1}: best={pop[0][2]:.6f}")

    pop.sort(key=lambda x: x[2])
    return pop[0]  # [cuts, times, fitness, chosen_times]

# ===== Run & report =====
best_cuts, best_times, best_f, best_chosen = run_ga()
best_cuts = repair(best_cuts)
segs = decode_groups(best_cuts)
t_star = T_GRID[best_chosen]

print("\n=== GA Result (5 bins, HARD constraint; BMI-grouping, Age in model) ===")
taus = []
for k, (s, e) in enumerate(segs):
    bmin, bmax = BMI_sorted[s], BMI_sorted[e-1]
    print(f"Group {k+1}: BMI in [{bmin:.2f}, {bmax:.2f}] -> GA={t_star[k]:.2f} wk  (n={e-s})")
    if k < K-1:
        next_min = BMI_sorted[segs[k+1][0]]
        taus.append(0.5*(bmax + next_min))
print("Cutpoints (taus):", [round(x, 2) for x in taus])

# Per-group stats（同时给出均值与硬比例，核验硬约束）
for k, (s, e) in enumerate(segs):
    j = best_chosen[k]
    mean_prob = Ptab[s:e, j].mean()
    prop_hard = (Ptab[s:e, j] >= PROB_REQ).mean()
    mean_risk = Rtab[s:e, j].mean()
    print(f"  Group{k+1}: mean P>4%={mean_prob:.3f}, prop(P>=0.8)={prop_hard:.3f}, mean risk={mean_risk:.3f}")

# Export per-individual recommendations (original order)
assign = np.empty(n, int)
for k, (s, e) in enumerate(segs): assign[s:e] = k
assign_orig = np.empty(n, int); assign_orig[order] = assign

out = pd.DataFrame({
    "BMI": BMI,
    "Age": AGE,
    "group": assign_orig + 1,
    "recommended_GA_week": t_star[assign_orig]
})
out.to_csv("nipt_ga_schedule.csv", index=False, encoding="utf-8-sig")
print("\nSaved: nipt_ga_schedule.csv")
